# Stats figure


In [ ]:
import pandas as pd
import numpy as np

from plotly.subplots import make_subplots
import plotly.graph_objects as go

from andeangc import config as cfg

In [ ]:
data = pd.read_csv(cfg.VERSION / 'AndeanGC_metadata.csv')

# Everything numeric is plotted, except the columns that are not basin attributes
exclude = ['gauge_lat', 'gauge_lon', 'days_w_data', 'days_w_data_qc']
attributes = [c for c in data.select_dtypes('number').columns if c not in exclude]

log_scale_attrs = ['basin_area', 'glacier_cover_RGI60', 'glacier_cover_RGI70', 'glacier_volume_M22', 'glacier_volume_F19', 'high_prec_freq_ERA5']

tickvals = [-3, -2, -1, 0, 1, 2, 3, 4]
ticktext = ["10<sup>-3</sup>", "10<sup>-2</sup>", "10<sup>-1</sup>", "10<sup>0</sup>", "10<sup>1</sup>", "10<sup>2</sup>", "10<sup>3</sup>", "10<sup>4</sup>"]

## Plot

In [ ]:
# Calculate subplot dimensions
n_cols = len(attributes)
n_rows = (n_cols + 5) // 6  # 6 columns per row

# Create subplots
fig = make_subplots(
    rows=n_rows,
    cols=6,
    subplot_titles=attributes,
    vertical_spacing=0.07,
    horizontal_spacing=0.03,
)

fig.for_each_annotation(lambda a: a.update(font=dict(size=13)))

# Create histogram for each attribute
for i, attr in enumerate(attributes):
    row = i // 6 + 1
    col_idx = i % 6 + 1
    
    # Use log scale for specified attributes
    x_data = np.log10(data[attr]) if attr in log_scale_attrs else data[attr]
    
    fig.add_trace(
        go.Histogram(x=x_data, name=attr, showlegend=False, nbinsx=30, marker_color="#3979a7"),
        row=row,
        col=col_idx,
    )
    
    # Set log scale tick labels if needed
    if attr in log_scale_attrs:
        fig.update_xaxes(tickvals=tickvals, ticktext=ticktext, row=row, col=col_idx)

# Update axes styling
fig.update_xaxes(griddash="dot", ticks="outside", title_standoff=5, tickfont=dict(size=10))
fig.update_yaxes(griddash="dot", ticks="outside", title_standoff=5, tickfont=dict(size=10))

# Update layout
fig.update_layout(
    height=900,
    width=1200,
    template="seaborn",
    margin=dict(l=10, r=10, b=10, t=10)
)

fig.show()
fig.write_image(cfg.VERSION / "figures" / 'figure02_stats.png', scale=4, height=900, width=1200)
